In [1]:
# Optimized VAE for missing-value imputation
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from scipy import stats
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# Check the TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# ==================== Enhanced reproducibility settings ====================
SEED = 42

def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Initialize random seeds
set_all_seeds(SEED)

# GPU memory settings
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Found {len(gpus)} GPU devices")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("No GPU found. Running on the CPU.")

# Clear the session
tf.keras.backend.clear_session()
gc.collect()

# 1. Data loading and preprocessing
def load_and_preprocess_data():
    """Load the data and perform initial preprocessing."""
    data = pd.read_csv("./Soil.csv", low_memory=True)
    print(f"Original data shape: {data.shape}")
    
    # Keep a copy of the original data
    original_data = data.copy()
    
    # Drop nonnumeric columns
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # Drop ID columns
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # Drop the L(m) column
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # Drop columns with more than 50% missing values
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.50].index
    data = data.drop(columns=cols_to_drop)
    
    # Convert data types
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"Final data shape: {data.shape}")
    print(f"Features: {list(data.columns)}")
    
    return data, original_data

# 2. Adaptive outlier treatment
def adaptive_outlier_handling(df):
    """Select an outlier treatment based on each feature's distribution."""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # Calculate skewness and kurtosis
        skewness = stats.skew(values)
        kurtosis = stats.kurtosis(values)
        
        # Choose the treatment based on the distribution
        if abs(skewness) > 2:  # Highly skewed
            # Use wider quantile cutoffs
            lower, upper = np.percentile(values, [0.1, 99.9])
        else:  # Approximately normal
            # Apply the IQR method
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr
            upper = q3 + 3 * iqr
            
            # Avoid discarding too much data
            lower = max(lower, np.percentile(values, 0.5))
            upper = min(upper, np.percentile(values, 99.5))
        
        values = np.clip(values, lower, upper)
        
        transform_info[column] = {
            'type': 'adaptive_clip',
            'lower': lower,
            'upper': upper,
            'skewness': skewness,
            'kurtosis': kurtosis
        }
        
        df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
        df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. Optimized mixed scaling
def optimized_normalization(data):
    """Scale each feature using a method suited to its distribution."""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # Calculate feature statistics
            skewness = stats.skew(values.flatten())
            value_range = values.max() - values.min()
            
            # Select the scaler for this feature
            if abs(skewness) > 1 or value_range > 1000:
                # Apply a quantile transform for high skewness or a wide range
                scaler = QuantileTransformer(
                    output_distribution='normal',
                    n_quantiles=min(1000, len(values)),
                    subsample=100000,
                    random_state=42
                )
            elif column in ['H(m)', 'At(m²）']:  # Features requiring separate treatment
                # Apply robust scaling
                scaler = RobustScaler(quantile_range=(5, 95))
            else:
                # Use standard scaling by default
                scaler = StandardScaler()
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # Map values to [0.1, 0.9]
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {
                'scaler': scaler,
                'min': min_val,
                'max': max_val,
                'type': type(scaler).__name__
            }
    
    return data_normalized, scalers

# 4. Improved inverse scaling
def inverse_transform_optimized(data, scalers, column_names):
    """Reverse the scaling."""
    data_original = np.zeros_like(data)
    
    for col_idx, col_name in enumerate(column_names):
        values = data[:, col_idx].copy()
        
        if col_name in scalers:
            scaler_info = scalers[col_name]
            
            # Reverse the range mapping
            values = (values - 0.1) / 0.8
            values = values * (scaler_info['max'] - scaler_info['min']) + scaler_info['min']
            
            # Reverse the scaling
            values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        data_original[:, col_idx] = values
    
    return data_original

# 5. Enhanced feature engineering
def enhanced_feature_engineering(data, mask):
    """Construct additional features."""
    n_samples, n_features = data.shape
    
    # Base features
    features = [data]
    
    # Squared features
    squared = data ** 2
    features.append(squared)
    
    # Square-root features with handling for negative values
    sqrt_data = np.sqrt(np.abs(data)) * np.sign(data)
    features.append(sqrt_data)
    
    # Log features with handling for zero and negative values
    log_data = np.log1p(np.abs(data)) * np.sign(data)
    features.append(log_data)
    
    # Construct selected feature interactions
    interactions = []
    for i in range(min(n_features, 3)):  # Limit the number of interactions
        for j in range(i+1, min(n_features, 3)):
            interaction = data[:, i:i+1] * data[:, j:j+1]
            interactions.append(interaction)
    
    if interactions:
        features.extend(interactions)
    
    # Combine all features
    enhanced_data = np.hstack(features)
    
    # Expand the mask
    n_new_features = enhanced_data.shape[1] - n_features
    enhanced_mask = np.hstack([mask] + [mask] * (n_new_features // n_features))
    
    # Resize the mask
    if enhanced_mask.shape[1] < enhanced_data.shape[1]:
        diff = enhanced_data.shape[1] - enhanced_mask.shape[1]
        enhanced_mask = np.hstack([enhanced_mask, mask[:, :diff]])
    
    return enhanced_data, enhanced_mask

# 6. Optimized VAE model
class FinalOptimizedVAE:
    """Optimized VAE for imputation."""
    def __init__(self, data_dim, enhanced_dim, latent_dim=8, learning_rate=0.0008, seed=42):
        self.data_dim = data_dim
        self.enhanced_dim = enhanced_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # Use cosine annealing with restarts
        lr_schedule = optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=learning_rate,
            first_decay_steps=500,
            t_mul=2.0,
            m_mul=0.9,
            alpha=0.1
        )
        self.optimizer = optimizers.Adam(learning_rate=lr_schedule)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
    
    def _build_encoder(self):
        """Build the optimized encoder."""
        inputs = layers.Input(shape=(self.enhanced_dim,))
        
        # Gradually reduce the number of dimensions
        x = layers.Dense(64, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(inputs)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.25)(x)
        
        x = layers.Dense(32, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(16, activation='swish',
                        kernel_initializer='he_normal')(x)
        x = layers.LayerNormalization()(x)
        
        # Latent space
        z_mean = layers.Dense(self.latent_dim)(x)
        z_log_var = layers.Dense(self.latent_dim)(x)
        
        # Bound the variance
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -4, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """Build the optimized decoder."""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        # Gradually increase the number of dimensions
        x = layers.Dense(16, activation='swish',
                        kernel_initializer='he_normal')(inputs)
        x = layers.LayerNormalization()(x)
        
        x = layers.Dense(32, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(64, activation='swish',
                        kernel_initializer='he_normal',
                        kernel_regularizer=regularizers.L2(0.003))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Dropout(0.25)(x)
        
        # Output layer
        outputs = layers.Dense(self.data_dim, activation='sigmoid')(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """Sample using the reparameterization trick."""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.05
    
    @tf.function
    def train_step(self, x_batch_enhanced, x_batch_original, mask_batch, beta):
        """Run one training step with KL annealing."""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(x_batch_enhanced, training=True)
            z = self._sample(z_mean, z_log_var)
            reconstructed = self.decoder(z, training=True)
            
            # Reconstruction loss
            mse = tf.square(x_batch_original - reconstructed) * mask_batch
            reconstruction_loss = tf.reduce_sum(mse) / tf.reduce_sum(mask_batch)
            
            # KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # Total loss with KL annealing
            total_loss = reconstruction_loss + beta * kl_loss
            
            # L2 regularization
            l2_loss = tf.add_n(self.encoder.losses + self.decoder.losses)
            total_loss += l2_loss
        
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        
        # Clip gradients
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss, reconstruction_loss, kl_loss
    
    def fit(self, data_enhanced, data_original, mask, validation_split=0.2, 
            epochs=120, batch_size=64):
        """Train the model with the configured learning rate and KL schedules."""
        n_samples = len(data_original)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train_enhanced = data_enhanced[train_indices]
        x_train_original = data_original[train_indices]
        x_val_enhanced = data_enhanced[val_indices]
        x_val_original = data_original[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"Training set: {len(x_train_original)} samples")
        print(f"Validation set: {len(x_val_original)} samples")
        
        patience = 25
        patience_counter = 0
        min_improvement = 0.00005
        
        # KL annealing schedule
        kl_anneal_epochs = 30
        
        for epoch in range(epochs):
            # KL annealing
            beta = min(1.0, epoch / kl_anneal_epochs) * 0.001
            
            train_indices_shuffled = np.random.permutation(len(x_train_original))
            train_loss = 0
            train_recon_loss = 0
            train_kl_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train_original), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss, recon_loss, kl_loss = self.train_step(
                    x_train_enhanced[batch_indices],
                    x_train_original[batch_indices],
                    mask_train[batch_indices],
                    beta
                )
                train_loss += loss
                train_recon_loss += recon_loss
                train_kl_loss += kl_loss
                n_batches += 1
            
            train_loss /= n_batches
            train_recon_loss /= n_batches
            train_kl_loss /= n_batches
            
            # Validation
            if epoch % 5 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val_original), batch_size):
                    z_mean, z_log_var = self.encoder(x_val_enhanced[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val_original[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                # Early-stopping logic
                if val_loss < self.best_val_loss - min_improvement:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 5
                
                if epoch % 20 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, "
                          f"Recon={train_recon_loss:.6f}, KL={train_kl_loss:.6f}, "
                          f"Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        
        # Restore the best weights
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data_enhanced, data_original, mask):
        """Impute missing values."""
        imputed = np.zeros_like(data_original)
        batch_size = 128
        
        for i in range(0, len(data_original), batch_size):
            batch_enhanced = data_enhanced[i:i+batch_size]
            batch_original = data_original[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # Average multiple draws
            predictions = []
            for _ in range(7):  # Increase the number of draws
                z_mean, z_log_var = self.encoder(batch_enhanced, training=False)
                z = self._sample(z_mean, z_log_var)
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            # Use a trimmed mean to reduce the influence of extreme samples
            predictions = np.array(predictions)
            pred_mean = np.mean(np.sort(predictions, axis=0)[1:-1], axis=0)
            
            imputed[i:i+batch_size] = batch_original * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 7. Enhanced feature-specific models
class EnhancedTargetedModel:
    """Enhanced models for specific features."""
    def __init__(self, target_features, seed=42):
        self.target_features = target_features
        self.seed = seed
        self.models = {}
    
    def fit(self, data_original, mask, column_names):
        """Train ensemble models for specific features."""
        print("Training enhanced feature-specific models...")
        
        simple_imputer = SimpleImputer(strategy='median')
        data_filled = simple_imputer.fit_transform(data_original)
        
        for col_idx, col_name in enumerate(column_names):
            if col_name not in self.target_features:
                continue
            
            obs_mask = mask[:, col_idx] == 1
            if np.sum(obs_mask) < 100:
                continue
            
            X_train = np.delete(data_filled[obs_mask], col_idx, axis=1)
            y_train = data_original[obs_mask, col_idx]
            
            # Train an ensemble model for each selected feature
            if col_name == 'H(m)':
                # Use a voting ensemble for H(m)
                models = [
                    ('mlp', MLPRegressor(
                        hidden_layer_sizes=(100, 50),
                        activation='tanh',
                        solver='lbfgs',
                        alpha=0.01,
                        max_iter=1000,
                        random_state=self.seed
                    )),
                    ('rf', RandomForestRegressor(
                        n_estimators=100,
                        max_depth=10,
                        random_state=self.seed
                    )),
                    ('gb', GradientBoostingRegressor(
                        n_estimators=100,
                        max_depth=5,
                        random_state=self.seed
                    ))
                ]
                model = VotingRegressor(models)
                
            elif col_name == 'At(m²）':
                # Use optimized ExtraTrees for At(m²）
                model = ExtraTreesRegressor(
                    n_estimators=200,
                    max_depth=12,
                    min_samples_split=3,
                    min_samples_leaf=1,
                    max_features='sqrt',
                    bootstrap=True,
                    random_state=self.seed
                )
                
            else:  # Wsa(m)
                # Use optimized gradient boosting for Wsa(m)
                model = GradientBoostingRegressor(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.03,
                    subsample=0.85,
                    min_samples_split=3,
                    min_samples_leaf=2,
                    random_state=self.seed
                )
            
            model.fit(X_train, y_train)
            self.models[col_idx] = model
            
            # Evaluate performance
            train_pred = model.predict(X_train)
            train_r2 = r2_score(y_train, train_pred)
            print(f"  {col_name}: training R²={train_r2:.4f}")
    
    def impute(self, data_original, mask):
        """Impute missing values."""
        simple_imputer = SimpleImputer(strategy='median')
        data_filled = simple_imputer.fit_transform(data_original)
        imputed = data_original.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(data_filled[miss_mask], col_idx, axis=1)
            imputed[miss_mask, col_idx] = model.predict(X_miss)
        
        return imputed

# 8. Final ensemble
class FinalEnsemble:
    """Final optimized ensemble."""
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = None
        self.scalers = None
        self.data_raw = None
    
    def fit(self, data_normalized, mask, data_raw, scalers, column_names):
        """Train the final ensemble."""
        print("\nTraining the final ensemble...")
        self.scalers = scalers
        self.data_raw = data_raw
        
        # Construct additional features
        data_enhanced, mask_enhanced = enhanced_feature_engineering(data_normalized, mask)
        print(f"Expanded feature count: {data_enhanced.shape[1]}")
        
        # 1. KNN imputer
        print("1. KNN imputer...")
        knn = KNNImputer(n_neighbors=12, weights='distance')
        knn.fit(data_normalized)
        self.models.append(('knn', knn))
        
        # 2. IterativeImputer
        print("2. IterativeImputer...")
        mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=50,
                max_depth=8,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        mice.fit(data_normalized)
        self.models.append(('mice', mice))
        
        # 3. Optimized VAE
        print("3. Optimized VAE model...")
        vae = FinalOptimizedVAE(
            data_dim=self.data_dim,
            enhanced_dim=data_enhanced.shape[1],
            latent_dim=8,
            learning_rate=0.0008,
            seed=self.seed
        )
        
        mean_imputer = SimpleImputer(strategy='median')
        mean_imputer.fit(data_normalized)
        initial_imputed = mean_imputer.transform(data_normalized)
        data_filled = np.where(mask == 1, data_normalized, initial_imputed)
        
        data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
        
        vae.fit(data_filled_enhanced, data_filled, mask, epochs=120, batch_size=64)
        self.models.append(('vae', vae))
        self.vae_enhanced_data = data_filled_enhanced
        
        # 4. Enhanced feature-specific models
        print("4. Enhanced feature-specific models...")
        target_model = EnhancedTargetedModel(
            target_features=['H(m)', 'At(m²）', 'Wsa(m)'],
            seed=self.seed
        )
        target_model.fit(data_raw, mask, column_names)
        self.models.append(('targeted', target_model))
        
        print("Ensemble training complete")
        
        # Adaptive weights
        self._compute_adaptive_weights()
    
    def _compute_adaptive_weights(self):
        """Calculate adaptive weights."""
        print("\nCalculating adaptive weights...")
        
        # Base weights
        base_weights = {
            'knn': 0.15,
            'mice': 0.20,
            'vae': 0.30,
            'targeted': 0.35
        }
        
        self.weights = np.array([base_weights[name] for name, _ in self.models])
        self.weights = self.weights / np.sum(self.weights)
        
        print(f"Final weights: {dict(zip([n for n, _ in self.models], self.weights))}")
    
    def impute(self, data_normalized, mask, column_names):
        """Predict using the final ensemble."""
        predictions = []
        
        for (name, model), weight in zip(self.models, self.weights):
            if name == 'knn':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'mice':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'vae':
                mean_imputer = SimpleImputer(strategy='median')
                mean_imputer.fit(data_normalized)
                initial_imputed = mean_imputer.transform(data_normalized)
                data_filled = np.where(mask == 1, data_normalized, initial_imputed)
                data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
                
                pred = model.impute(data_filled_enhanced, data_filled, mask)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'targeted':
                pred_original = model.impute(self.data_raw, mask)
            
            predictions.append(pred_original * weight)
        
        # Weighted average
        imputed = np.sum(predictions, axis=0)
        
        # Preserve observed values
        imputed = self.data_raw * mask + imputed * (1 - mask)
        
        return imputed

# 9. Final evaluation
# Export observed and predicted values to CSV
# ============================================================
def final_evaluation(model, data_normalized, mask, column_names, 
                     data_raw, scalers):
    """Evaluate the model and export observed and predicted values for each feature."""
    print("\n=== Final evaluation (5-fold cross-validation) ===")
    
    results = []
    output_dir = "observed_vs_predicted_soil"
    os.makedirs(output_dir, exist_ok=True)  # Create the output directory

    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        
        all_true_original = []
        all_pred_original = []
        fold_metrics = []
        
        for fold, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            test_indices = obs_indices[test_idx]
            
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            imputed_original = model.impute(data_normalized, test_mask, column_names)
            
            true_values = data_raw[test_indices, col_idx]
            pred_values = imputed_original[test_indices, col_idx]
            
            if len(true_values) > 1:
                fold_r2 = r2_score(true_values, pred_values)
                fold_mae = mean_absolute_error(true_values, pred_values)
                fold_rmse = np.sqrt(mean_squared_error(true_values, pred_values))  # Calculate RMSE
                fold_metrics.append({'r2': fold_r2, 'mae': fold_mae, 'rmse': fold_rmse})
            
            all_true_original.extend(true_values)
            all_pred_original.extend(pred_values)
        
        all_true_original = np.array(all_true_original)
        all_pred_original = np.array(all_pred_original)
        
        # Save observed and predicted values
        compare_df = pd.DataFrame({
            'True': all_true_original,
            'Pred': all_pred_original
        })
        safe_name = str(col_name).replace('/', '_').replace('(', '').replace(')', '').replace('²', '2')
        compare_path = os.path.join(output_dir, f"observed_vs_predicted_{safe_name}.csv")
        compare_df.to_csv(compare_path, index=False, encoding='utf-8-sig')

        # Calculate overall metrics
        mae = mean_absolute_error(all_true_original, all_pred_original)
        rmse = np.sqrt(mean_squared_error(all_true_original, all_pred_original))  # Calculate RMSE
        ss_res = np.sum((all_true_original - all_pred_original) ** 2)
        ss_tot = np.sum((all_true_original - np.mean(all_true_original)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        r2_scores = [m['r2'] for m in fold_metrics]
        r2_std = np.std(r2_scores) if r2_scores else 0
        
        non_zero_mask = np.abs(all_true_original) > 1e-8
        if np.sum(non_zero_mask) > 0:
            mape = np.mean(np.abs((all_true_original[non_zero_mask] - all_pred_original[non_zero_mask]) / 
                                all_true_original[non_zero_mask])) * 100
        else:
            mape = np.mean(np.abs(all_true_original - all_pred_original)) * 100
        
        if np.std(all_true_original) > 0 and np.std(all_pred_original) > 0:
            correlation = np.corrcoef(all_true_original, all_pred_original)[0, 1]
        else:
            correlation = 0
        
        status = "Excellent"
        if r2 < 0.6:
            status = "Needs improvement"
        elif r2 < 0.8:
            status = "Good"
        elif r2 > 0.95 and mape > 30:
            status = "Check for overfitting"
        elif r2_std > 0.05:
            status = "Stability needs improvement"
        
        results.append({
            'Feature': col_name,
            'R²': r2,
            'R²_std': r2_std,
            'Correlation': correlation,
            'MAE': mae,
            'RMSE': rmse,  # Include RMSE
            'MAPE(%)': mape,
            'Status': status
        })
        
        print(f"{col_name}: R²={r2:.4f}±{r2_std:.4f}, MAPE={mape:.2f}%, status={status}, RMSE={rmse:.4f}")
        print(f"  ↳ Saved observed and predicted values to {compare_path}")
    
    return pd.DataFrame(results)

# 10. Main function
def main():
    """Run the workflow."""
    print("=" * 80)
    print("Optimized VAE for missing-value imputation")
    print("Target: R² ≥ 0.8 for every feature, with stable model performance")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. Load the data
        print("\n1. Loading data...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 2. Adaptive outlier treatment
        print("\n2. Applying adaptive outlier treatment...")
        data_clean, transform_info = adaptive_outlier_handling(data_raw)
        
        # 3. Optimized scaling
        print("\n3. Applying optimized mixed scaling...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # Create the mask
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        data_raw_array = data_clean.fillna(0).values.astype(np.float32)
        
        print(f"\nData shape: {data_array.shape}")
        print(f"Missing rate: {(1 - mask.mean())*100:.2f}%")
        
        # Report the scaling methods
        print("\nScaling methods:")
        for col_name, scaler_info in scalers.items():
            print(f"  {col_name}: {scaler_info['type']}")
        
        # 4. Train the final ensemble
        print("\n4. Training the final ensemble...")
        model = FinalEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_raw_array, scalers, data_normalized.columns)
        
        # 5. Final evaluation
        print("\n5. Running the final evaluation...")
        eval_results = final_evaluation(
            model, data_array, mask, data_normalized.columns,
            data_raw_array, scalers
        )
        
        # 6. Generate the imputed data
        print("\n6. Generating the final imputed data...")
        final_imputed_original = model.impute(data_array, mask, data_normalized.columns)
        
        # 7. Report the results
        print("\n" + "=" * 80)
        print("Final model performance")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['Feature']}:")
            print(f"  R² = {row['R²']:.4f} ± {row['R²_std']:.4f}")
            print(f"  MAE = {row['MAE']:.4f}")
            print(f"  MAPE = {row['MAPE(%)']:.2f}%")
            print(f"  Status: {row['Status']}")
        
        # Summary statistics
        print("\n" + "-" * 40)
        print("Summary statistics")
        print("-" * 40)
        
        avg_r2 = eval_results['R²'].mean()
        min_r2 = eval_results['R²'].min()
        max_r2 = eval_results['R²'].max()
        avg_mape = eval_results['MAPE(%)'].mean()
        
        print(f"Mean R²: {avg_r2:.4f}")
        print(f"Minimum R²: {min_r2:.4f}")
        print(f"Maximum R²: {max_r2:.4f}")
        print(f"Mean MAPE: {avg_mape:.2f}%")
        
        # Check whether the features meet the target
        target_r2 = 0.8
        achieved = eval_results[eval_results['R²'] >= target_r2]
        not_achieved = eval_results[eval_results['R²'] < target_r2]
        
        print(f"\nTarget check (R² ≥ {target_r2}):")
        print(f"  Features meeting the target: {len(achieved)}/{len(eval_results)}")
        
        if not_achieved.empty:
            print("  ✅ All features meet the target")
        else:
            print("  Features below the target:")
            for _, row in not_achieved.iterrows():
                print(f"    - {row['Feature']}: R²={row['R²']:.4f}")
        
        # Check stability
        unstable = eval_results[eval_results['R²_std'] > 0.05]
        if not unstable.empty:
            print("\nStability warning (R²_std > 0.05):")
            for _, row in unstable.iterrows():
                print(f"  - {row['Feature']}: std={row['R²_std']:.4f}")
        else:
            print("\n✅ Stable performance across all features")
        
        # Save the results
        final_df_original = pd.DataFrame(final_imputed_original, columns=data_normalized.columns)
        final_df_original.to_csv("imputed_soil.csv", index=False)
        eval_results.to_csv("evaluation_soil.csv", index=False)
        
        print("\nFiles saved:")
        print("- imputed_soil.csv")
        print("- evaluation_soil.csv")
        
        return final_imputed_original, eval_results
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    print("\nStarting the workflow...")
    print("Model settings:")
    print("1. Adaptive outlier treatment based on skewness and kurtosis")
    print("2. Scaling selected by feature distribution")
    print("3. Squared, square-root, log, and interaction features")
    print("4. Swish activation and layer normalization")
    print("5. KL annealing and a cosine learning-rate schedule")
    print("6. Feature-specific ensemble models")
    print("7. Trimmed means to reduce the effect of extreme samples")
    print("8. MAPE calculation with handling for zero values")
    
    start_time = time.time()
    result, evaluation = main()
    
    if result is not None:
        print(f"\n✅ Run complete!")
        print(f"Total time: {(time.time() - start_time)/60:.2f} minutes")
        
        # Performance summary
        if evaluation is not None:
            print("\n" + "=" * 50)
            print("Performance summary")
            print("=" * 50)
            
            if evaluation['R²'].min() >= 0.8:
                print("✅ All features meet the R² target of 0.8")
            
            if evaluation['R²_std'].max() < 0.05:
                print("✅ Excellent model stability")
            
            if evaluation['MAPE(%)'].max() < 50:
                print("✅ Relative error meets the configured threshold")
            
            print("\nThank you for using this notebook!")
    else:
        print("\n❌ Run failed. See the error above.")
    
    gc.collect()

TensorFlow version: 2.21.0
No GPU found. Running on the CPU.


Starting the workflow...
Model settings:
1. Adaptive outlier treatment based on skewness and kurtosis
2. Scaling selected by feature distribution
3. Squared, square-root, log, and interaction features
4. Swish activation and layer normalization
5. KL annealing and a cosine learning-rate schedule
6. Feature-specific ensemble models
7. Trimmed means to reduce the effect of extreme samples
8. MAPE calculation with handling for zero values
Optimized VAE for missing-value imputation
Target: R² ≥ 0.8 for every feature, with stable model performance

1. Loading data...

Error: [Errno 2] No such file or directory: './Soil.csv'

❌ Run failed. See the error above.


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27516\1920333375.py", line 861, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27516\1920333375.py", line 62, in load_and_preprocess_data
    data = pd.read_csv("./Soil.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, in 

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    Add the L(m) column to the imputed results.
    """
    # 1. Read the source data to retrieve L(m)
    original_data = pd.read_csv("./Soil.csv", low_memory=True)
    
    # 2. Check whether L(m) is present
    if 'L(m)' not in original_data.columns:
        print("The source data does not contain L(m)")
        return
    
    # 3. Extract L(m)
    L_column = original_data['L(m)'].copy()
    print(f"Extracted L(m) with {len(L_column)} rows")
    
    # 4. Read the imputed results
    try:
        imputed_results = pd.read_csv("imputed_soil.csv")
        print(f"Imputed data shape: {imputed_results.shape}")
    except FileNotFoundError:
        print("Imputed results file not found. Run the imputation cell first.")
        return
    
    # 5. Check that the row counts match
    if len(imputed_results) != len(L_column):
        print(f"Warning: imputed results have {len(imputed_results)} rows, while L(m) has {len(L_column)} rows")
        # If row counts differ, align both tables to the shorter length
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. Add L(m) to the results
    imputed_results['L(m)'] = L_column.values
    
    # 7. Save the updated results
    output_filename = "imputed_soil_with_runout.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\nAdded L(m) to the imputed results.")
    print(f"Saved the new file as: {output_filename}")
    print(f"Final data shape: {imputed_results.shape}")
    print(f"Column names: {list(imputed_results.columns)}")
    
    # 8. Report summary statistics for L(m)
    print("\nL(m) summary statistics:")
    print(f"  Data type: {L_column.dtype}")
    print(f"  Unique values: {L_column.nunique()}")
    print(f"  Missing values: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  Minimum: {L_column.min()}")
        print(f"  Maximum: {L_column.max()}")
        print(f"  Mean: {L_column.mean():.2f}")
    else:
        print(f"  First five values: {L_column.head().tolist()}")
    
    return imputed_results

# Run the function
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

Extracted L(m) with 4349 rows
Imputed data shape: (4349, 5)

Added L(m) to the imputed results.
Saved the new file as: imputed_soil_with_runout.csv
Final data shape: (4349, 6)
Column names: ['H(m)', 'As(m²）', 'At(m²）', 'V(m³)', 'Wsa(m)', 'L(m)']

L(m) summary statistics:
  Data type: float64
  Unique values: 3107
  Missing values: 0
  Minimum: 1.17
  Maximum: 1610000.0
  Mean: 760.89
